# NB31: USGS λ-vs-β Stratified by Contamination Level

Tests whether the λ-β relationship identified in NB30 (phylogenetic conservation
predicts metal association strength) is contingent on the local contamination
context. The mechanistic hypothesis: if λ reflects genuine adaptation under
metal selection pressure, then more conserved KOs should associate more strongly
with metals *specifically* in high-contamination environments.

**Design:**
- For each USGS metal, genera are split into tertiles (low/mid/high) by their
  mean metal concentration across all USGS sites where they appear.
- Within each stratum, the same PGLS sweep from NB30 is repeated:
  PGLS(log_usgs_ppm ~ ko_density_z + controls) across 35 KOs × 12 control combos.
- Spearman ρ(λ_uncond, |β|) is computed per metal × stratum × control.
- Key test: is ρ_high > ρ_low? (monotone increase = suppressor/selection hypothesis)

**Output files:**
- `nb31_usgs_stratum_sweep.parquet` — all per-KO PGLS results
- `nb31_stratum_rho_summary.csv` — Spearman ρ per metal × stratum × control
- `nb31_monotonicity_test.csv` — Δρ (high − low) per metal × control
- Figures: F1 heatmap, F2 trajectory, F3 monotonicity summary

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT      = Path('/home/hmacgregor/BERIL-research-observatory')
DATA      = ROOT / 'projects/comprehensive_metal_ecology/data'
FIGS      = ROOT / 'projects/comprehensive_metal_ecology/figures'
TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(ROOT / 'tools'))
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

sys.path.insert(0, str(ROOT / 'projects/comprehensive_metal_ecology/scripts'))
from pgls_utils import load_tree, build_vcv, _optimise_lambda, _gls_fit

print('Setup done.')

Setup done.


In [2]:
print('Loading data...')

phylo_lam  = pd.read_csv(DATA / 'phylo_d_all_ko.csv')
curated    = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
genus_env  = pd.read_csv(DATA / 'genus_lat_env_covariates.csv')
spark      = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')
genus_usgs = pd.read_csv(DATA / 'nb27_genus_usgs_means.csv')
csu_beta   = pd.read_csv(DATA / 'per_ko_lambda_csu_mobility.csv')

nb25 = pd.read_parquet(DATA / 'nb25_ko_presence_matrix.parquet')
nb25['genus_lower'] = nb25['genus_lower'].str.replace('g__', '', regex=False)

tier12     = curated[curated['evidence_tier'].isin(['Tier 1', 'Tier 2'])]
FITTED_KOS = sorted(csu_beta['ko_id'].unique())

density_base = (
    nb25[nb25['ko'].isin(FITTED_KOS)]
    .merge(spark[['genus_lower', 'n_genomes', 'mean_genome_mb']], on='genus_lower', how='inner')
)
density_base['density'] = (
    density_base['n_genomes_with_ko'] /
    (density_base['n_genomes'] * density_base['mean_genome_mb'])
)

tree = load_tree(str(TREE_PATH))
tree_labels = {t.label.replace(' ', '_').lower() for t in tree.taxon_namespace}
print(f'Tree loaded: {len(tree_labels):,} taxa')

USGS_RAW_COLS = [c for c in genus_usgs.columns if c.startswith('mean_usgs_raw_')]
USGS_METALS   = [c.replace('mean_usgs_raw_', '') for c in USGS_RAW_COLS]
print(f'USGS metals: {USGS_METALS}')
print(f'Fitted KOs:  {len(FITTED_KOS)}')
print(f'Genera in genus_usgs: {len(genus_usgs):,}')

Loading data...
Tree loaded: 2,283 taxa
USGS metals: ['as', 'cd', 'cr', 'cu', 'ni', 'pb', 'zn']
Fitted KOs:  35
Genera in genus_usgs: 3,239


In [3]:
def zscore(x):
    mu, sd = np.nanmean(x), np.nanstd(x, ddof=1)
    return (x - mu) / sd if sd > 0 else x - mu


def run_pgls_fast(df, response_col, predictor_cols, taxon_col='genus_lower', min_n=30):
    keep = df[taxon_col].str.replace(' ', '_').str.lower().isin(tree_labels) & df[response_col].notna()
    for pc in predictor_cols:
        keep = keep & df[pc].notna()
    sub = df[keep].copy()
    sub['_taxon'] = sub[taxon_col].str.replace(' ', '_').str.lower()
    sub = sub.drop_duplicates('_taxon')
    n = len(sub)
    if n < min_n:
        return None
    taxa = sub['_taxon'].tolist()
    y    = sub[response_col].values.astype(float)
    X    = np.column_stack([np.ones(n)] + [sub[pc].values.astype(float) for pc in predictor_cols])
    V    = build_vcv(tree, taxa)
    lam, _ = _optimise_lambda(y, X, V)
    ll, sigma2, betas, betas_se, _, _ = _gls_fit(y, X, V, lam)
    t_stats  = betas / np.where(betas_se > 0, betas_se, np.nan)
    df_resid = n - len(betas)
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=df_resid)
    return dict(n=n, lambda_est=lam,
                beta=betas[1] if len(betas) > 1 else betas[0],
                SE=betas_se[1] if len(betas_se) > 1 else betas_se[0],
                p_value=p_values[1] if len(p_values) > 1 else p_values[0])


CONTROL_COMBOS = [
    ('None',                       []),
    ('pH',                         ['ph_z']),
    ('SOM',                        ['som_z']),
    ('Temp',                       ['temp_z']),
    ('pH + SOM',                   ['ph_z', 'som_z']),
    ('pH + Temp',                  ['ph_z', 'temp_z']),
    ('SOM + Temp',                 ['som_z', 'temp_z']),
    ('pH + SOM + Temp',            ['ph_z', 'som_z', 'temp_z']),
    ('|Lat|',                      ['lat_z']),
    ('pH + |Lat|',                 ['ph_z', 'lat_z']),
    ('pH + SOM + |Lat|',           ['ph_z', 'som_z', 'lat_z']),
    ('pH + SOM + Temp + |Lat|',    ['ph_z', 'som_z', 'temp_z', 'lat_z']),
]
COMBO_LABELS   = [c[0] for c in CONTROL_COMBOS]
STRATUM_LABELS = ['low', 'mid', 'high']
STRATUM_RANK   = {'low': 0, 'mid': 1, 'high': 2}

print(f'{len(CONTROL_COMBOS)} control combinations, {len(STRATUM_LABELS)} strata defined.')

12 control combinations, 3 strata defined.


In [4]:
# Cut each metal into equal-count tertiles; store genus → stratum mapping
print('Computing contamination tertiles per metal...')

stratum_maps = {}
for metal in USGS_METALS:
    metal_col = f'mean_usgs_raw_{metal}'
    metal_df  = genus_usgs[['genus_lower', metal_col]].dropna().copy()
    metal_df['stratum'] = pd.qcut(
        metal_df[metal_col], q=3, labels=STRATUM_LABELS
    )
    stratum_maps[metal] = metal_df.set_index('genus_lower')['stratum'].astype(str).to_dict()
    counts = metal_df['stratum'].value_counts().reindex(STRATUM_LABELS)
    ppm_bounds = metal_df.groupby('stratum')[metal_col].agg(['min', 'max'])
    print(f'  {metal.upper():3s}:  ' +
          '  '.join(f'{s}={counts[s]} ({ppm_bounds.loc[s, "min"]:.2g}–{ppm_bounds.loc[s, "max"]:.2g} ppm)'
                    for s in STRATUM_LABELS))

Computing contamination tertiles per metal...


  AS :  low=1069 (0.35–4.9 ppm)  mid=1073 (4.9–8.3 ppm)  high=1061 (8.4–1.1e+02 ppm)
  CD :  low=1011 (0.05–0.29 ppm)  mid=909 (0.3–0.56 ppm)  high=952 (0.57–5.4 ppm)
  CR :  low=1080 (1–24 ppm)  mid=1459 (24–41 ppm)  high=698 (41–1.4e+03 ppm)
  CU :  low=1096 (0.79–16 ppm)  mid=1195 (16–26 ppm)  high=908 (26–2.4e+02 ppm)
  NI :  low=1466 (1.3–15 ppm)  mid=907 (15–19 ppm)  high=853 (19–1.3e+03 ppm)
  PB :  low=1098 (3.5–18 ppm)  mid=1279 (18–72 ppm)  high=859 (72–3.3e+02 ppm)
  ZN :  low=1117 (4–54 ppm)  mid=1055 (54–78 ppm)  high=1066 (78–6.5e+02 ppm)


In [5]:
# PGLS sweep: 35 KOs × 7 metals × 3 strata × 12 control combos
SWEEP_PATH = DATA / 'nb31_usgs_stratum_sweep.parquet'

if SWEEP_PATH.exists():
    print('Loading cached sweep...')
    sweep = pd.read_parquet(SWEEP_PATH)
else:
    n_expected = len(FITTED_KOS) * len(USGS_METALS) * len(STRATUM_LABELS) * len(CONTROL_COMBOS)
    print(f'Running PGLS sweep: {len(FITTED_KOS)} KOs × {len(USGS_METALS)} metals '
          f'× {len(STRATUM_LABELS)} strata × {len(CONTROL_COMBOS)} combos')
    print(f'  = up to {n_expected:,} fits (fewer if n<30 in stratum)')

    env_lookup = genus_env[['genus_lower', 'median_soil_ph', 'median_soil_som',
                             'median_era5_temp_C', 'lat_abs']].dropna().copy()

    rows    = []
    n_ko    = len(FITTED_KOS)

    for i_ko, ko_id in enumerate(FITTED_KOS):
        if (i_ko + 1) % 5 == 0 or i_ko == 0:
            print(f'  KO {i_ko+1}/{n_ko}: {ko_id}', flush=True)

        density_ko = density_base[density_base['ko'] == ko_id][['genus_lower', 'density']].copy()
        if len(density_ko) == 0:
            continue

        merged = (
            density_ko
            .merge(genus_usgs, on='genus_lower', how='inner')
            .merge(env_lookup, on='genus_lower', how='inner')
        )

        meta_rows   = tier12[tier12['KO'] == ko_id]
        gene_name   = meta_rows['gene_name'].values[0]        if len(meta_rows) > 0 else ''
        subcategory = meta_rows['primary_category'].values[0] if len(meta_rows) > 0 else ''

        for metal in USGS_METALS:
            metal_col = f'mean_usgs_raw_{metal}'
            needed    = [metal_col, 'density', 'median_soil_ph', 'median_soil_som',
                         'median_era5_temp_C', 'lat_abs']
            sub_all = merged.dropna(subset=needed).copy()
            if len(sub_all) < 30:
                continue

            sub_all['stratum'] = sub_all['genus_lower'].map(stratum_maps[metal])
            sub_all = sub_all.dropna(subset=['stratum'])

            for stratum in STRATUM_LABELS:
                sub = sub_all[sub_all['stratum'] == stratum].copy()
                if len(sub) < 20:
                    continue

                sub['metal_log'] = np.log1p(sub[metal_col])
                sub['metal_z']   = zscore(sub['metal_log'])
                sub['density_z'] = zscore(sub['density'])
                sub['ph_z']      = zscore(sub['median_soil_ph'])
                sub['som_z']     = zscore(sub['median_soil_som'])
                sub['temp_z']    = zscore(sub['median_era5_temp_C'])
                sub['lat_z']     = zscore(sub['lat_abs'])

                for combo_label, extra_preds in CONTROL_COMBOS:
                    predictors = ['density_z'] + extra_preds
                    result = run_pgls_fast(sub, 'metal_z', predictors)
                    if result:
                        rows.append({
                            'ko_id': ko_id, 'gene_name': gene_name,
                            'subcategory': subcategory,
                            'metal': metal, 'stratum': stratum,
                            'controls': combo_label,
                            'n': result['n'],
                            'lambda_pgls': result['lambda_est'],
                            'beta': result['beta'],
                            'SE': result['SE'],
                            'p_value': result['p_value'],
                        })

    sweep = pd.DataFrame(rows)
    sweep.attrs = {}
    sweep.to_parquet(SWEEP_PATH, index=False)
    print(f'Saved {len(sweep):,} rows to {SWEEP_PATH.name}')

print(f'Sweep rows: {len(sweep):,}')
print(f'KOs fitted: {sweep["ko_id"].nunique()}')
print(sweep.groupby(['metal', 'stratum'])['ko_id'].nunique().unstack())

Loading cached sweep...
Sweep rows: 4,536
KOs fitted: 26
stratum  high  low  mid
metal                  
as         20   19   15
cd         15   19   14
cr         12   19   26
cu         16   20   19
ni         15   23   14
pb         15   19   22
zn         21   20   15


In [6]:
# Merge unconditional λ into sweep results
phylo_lam_sub = (
    phylo_lam[phylo_lam['ko_id'].isin(FITTED_KOS)]
    [['ko_id', 'lambda']]
    .rename(columns={'lambda': 'lambda_uncond'})
)

sweep_lam = sweep.merge(phylo_lam_sub, on='ko_id', how='inner')
sweep_lam['abs_beta']     = np.abs(sweep_lam['beta'])
sweep_lam['sig']          = sweep_lam['p_value'] < 0.05
sweep_lam['stratum_rank'] = sweep_lam['stratum'].map(STRATUM_RANK)

print(f'Joined rows: {len(sweep_lam):,}')
print(f'KOs with λ: {sweep_lam["ko_id"].nunique()}')

Joined rows: 4,536
KOs with λ: 26


In [7]:
# Spearman ρ(λ_uncond, |β|) per metal × stratum × control
summary_rows = []

for (metal, stratum, ctrl), grp in sweep_lam.groupby(['metal', 'stratum', 'controls']):
    if len(grp) < 5:
        continue
    rho, p = stats.spearmanr(grp['lambda_uncond'], grp['abs_beta'])
    summary_rows.append({
        'metal': metal, 'stratum': stratum, 'controls': ctrl,
        'stratum_rank': STRATUM_RANK[stratum],
        'rho': rho, 'p_value': p,
        'n_kos': len(grp), 'n_sig_kos': grp['sig'].sum(),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(DATA / 'nb31_stratum_rho_summary.csv', index=False)
print(f'Saved nb31_stratum_rho_summary.csv ({len(summary)} rows)')

# Quick overview: ρ for None controls per metal × stratum
print('\nρ(λ_uncond, |β|) for "None" controls — low / mid / high:')
none_summary = summary[summary['controls'] == 'None'].copy()
pivot_none = none_summary.pivot(index='metal', columns='stratum', values='rho')[STRATUM_LABELS]
pivot_none['Δ (high−low)'] = pivot_none['high'] - pivot_none['low']
print(pivot_none.round(3).to_string())

Saved nb31_stratum_rho_summary.csv (252 rows)

ρ(λ_uncond, |β|) for "None" controls — low / mid / high:


stratum    low    mid   high  Δ (high−low)
metal                                     
as      -0.560  0.079  0.020         0.579
cd       0.212 -0.270 -0.418        -0.630
cr      -0.030  0.142 -0.028         0.002
cu      -0.188  0.095 -0.365        -0.177
ni       0.216  0.226 -0.471        -0.688
pb       0.300 -0.019  0.043        -0.257
zn       0.171 -0.146  0.596         0.425


In [8]:
# Does ρ increase monotonically with contamination level?
mono_rows = []

for (metal, ctrl), grp in summary.groupby(['metal', 'controls']):
    grp_s = grp.set_index('stratum').reindex(STRATUM_LABELS)
    rhos = grp_s['rho'].values
    ns   = grp_s['n_kos'].values
    if np.any(pd.isna(rhos)):
        continue
    strictly_up = bool((rhos[1] > rhos[0]) and (rhos[2] > rhos[1]))
    delta = float(rhos[2] - rhos[0])
    mono_rows.append({
        'metal': metal, 'controls': ctrl,
        'rho_low': rhos[0], 'rho_mid': rhos[1], 'rho_high': rhos[2],
        'delta_high_minus_low': delta,
        'strictly_monotone': strictly_up,
        'n_low': int(ns[0]), 'n_mid': int(ns[1]), 'n_high': int(ns[2]),
    })

mono_df = pd.DataFrame(mono_rows)
mono_df.to_csv(DATA / 'nb31_monotonicity_test.csv', index=False)

n_mono  = mono_df['strictly_monotone'].sum()
n_total = len(mono_df)
mean_d  = mono_df['delta_high_minus_low'].mean()

print(f'Strictly monotone (ρ_low < ρ_mid < ρ_high): {n_mono}/{n_total} ({100*n_mono/n_total:.1f}%)')
print(f'Random chance:  33.3% (1/3 of orderings are strictly increasing)')
print(f'Mean Δρ (high − low) across all metal×combo: {mean_d:+.3f}')

print('\nMonotonicity by metal (across 12 control combos):')
print(f'{"Metal":5s}  {"% Monotone":>11s}  {"Mean Δρ":>9s}')
print('-' * 32)
for metal in sorted(mono_df['metal'].unique()):
    sub = mono_df[mono_df['metal'] == metal]
    pct = 100 * sub['strictly_monotone'].mean()
    md  = sub['delta_high_minus_low'].mean()
    print(f'{metal.upper():5s}  {pct:>10.0f}%  {md:>+9.3f}')

Strictly monotone (ρ_low < ρ_mid < ρ_high): 9/84 (10.7%)
Random chance:  33.3% (1/3 of orderings are strictly increasing)
Mean Δρ (high − low) across all metal×combo: -0.156

Monotonicity by metal (across 12 control combos):
Metal   % Monotone    Mean Δρ
--------------------------------
AS             67%     +0.531
CD              0%     -0.408
CR              8%     -0.014
CU              0%     -0.409
NI              0%     -0.439
PB              0%     -0.475
ZN              0%     +0.120


In [9]:
# Figure 1: 3-panel heatmap — ρ by metal × stratum for 3 key control combos
KEY_CONTROLS = ['None', 'pH + SOM + Temp', 'pH + SOM + Temp + |Lat|']
metals_order = sorted(USGS_METALS)

fig, axs = plt.subplots(1, 3, figsize=(FIGW['full'], ROW_H * 1.4),
                        sharey=True)

vmax = 0.65

for ax, ctrl in zip(axs, KEY_CONTROLS):
    sub = summary[summary['controls'] == ctrl]
    pivot = sub.pivot(index='metal', columns='stratum', values='rho').reindex(
        index=metals_order, columns=STRATUM_LABELS)
    pivot_p = sub.pivot(index='metal', columns='stratum', values='p_value').reindex(
        index=metals_order, columns=STRATUM_LABELS)

    im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax)

    for i, metal in enumerate(metals_order):
        for j, stratum in enumerate(STRATUM_LABELS):
            rho_val = pivot.iloc[i, j]
            p_val   = pivot_p.iloc[i, j]
            if pd.isna(rho_val):
                continue
            marker = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else '')
            txt    = f'{rho_val:+.2f}{marker}'
            color  = 'white' if abs(rho_val) > 0.38 else 'black'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color=color,
                    fontweight='bold' if marker else 'normal')

    ax.set_xticks(range(3))
    ax.set_xticklabels(['Low\nMetal', 'Mid\nMetal', 'High\nMetal'], fontsize=8)
    ax.set_yticks(range(len(metals_order)))
    ax.set_yticklabels([m.upper() for m in metals_order], fontsize=8)
    ax.set_title(ctrl, fontsize=9)
    ax.set_xlabel('Contamination stratum', fontsize=9)

axs[0].set_ylabel('USGS metal', fontsize=9)

plt.colorbar(im, ax=axs[-1], label='Spearman ρ (λ ~ |β|)', shrink=0.7)
fig.suptitle('NB31 Fig 1: λ-β ρ by contamination stratum and control combination\n'
             '(*p<0.05, **p<0.01)', y=1.02)
save(fig, FIGS / 'nb31_F1_stratum_heatmap')
print('Saved nb31_F1_stratum_heatmap.pdf')

Saved nb31_F1_stratum_heatmap.pdf


In [10]:
# Figure 2: Per-metal trajectory — ρ vs stratum for each control combo
n_metals = len(USGS_METALS)
n_cols   = 4
n_rows   = (n_metals + n_cols - 1) // n_cols

fig, axs = plt.subplots(n_rows, n_cols,
                        figsize=(FIGW['full'], ROW_H * n_rows),
                        sharey=True)
axs_flat = axs.flatten()

x_pos    = [0, 1, 2]
ctrl_alpha = {c: (0.9 if c in ('None', 'pH + SOM + Temp + |Lat|') else 0.35)
              for c in COMBO_LABELS}
ctrl_lw    = {c: (2.0 if c in ('None', 'pH + SOM + Temp + |Lat|') else 0.8)
              for c in COMBO_LABELS}

for ax, metal in zip(axs_flat, sorted(USGS_METALS)):
    sub_metal = summary[summary['metal'] == metal]

    for i_ctrl, ctrl in enumerate(COMBO_LABELS):
        sub_ctrl = sub_metal[sub_metal['controls'] == ctrl].set_index('stratum').reindex(STRATUM_LABELS)
        rhos = sub_ctrl['rho'].values
        ps   = sub_ctrl['p_value'].values
        col  = PALETTE[i_ctrl % len(PALETTE)]
        lw   = ctrl_lw[ctrl]
        al   = ctrl_alpha[ctrl]
        label = ctrl if ctrl in ('None', 'pH + SOM + Temp + |Lat|') else None
        ax.plot(x_pos, rhos, '-o', color=col, lw=lw, alpha=al, ms=4, label=label)
        # mark significant points with black ring
        for xi, (r, p) in enumerate(zip(rhos, ps)):
            if not np.isnan(r) and p < 0.05:
                ax.plot(xi, r, 'o', color=col, ms=8,
                        markeredgecolor='k', markeredgewidth=0.8)

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Low', 'Mid', 'High'], fontsize=8)
    ax.set_xlabel('Contamination stratum', fontsize=9)
    ax.set_ylabel('Spearman ρ (λ ~ |β|)', fontsize=9)
    ax.set_title(metal.upper(), fontsize=10)
    grid_h(ax)

axs_flat[0].legend(fontsize=7, loc='best')

for ax in axs_flat[n_metals:]:
    ax.set_visible(False)

fig.suptitle('NB31 Fig 2: λ-β ρ trajectory across contamination strata\n'
             '(bold = None / full panel; filled = p<0.05)', y=1.02)
save(fig, FIGS / 'nb31_F2_stratum_trajectory')
print('Saved nb31_F2_stratum_trajectory.pdf')

Saved nb31_F2_stratum_trajectory.pdf


In [11]:
# Figure 3: Monotonicity summary
# Panel A: mean Δρ (high−low) per metal, violin of all 12 combos
# Panel B: % strictly monotone per metal (bar chart)

metals_sorted = sorted(mono_df['metal'].unique())

fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Panel A: Δρ distribution per metal
ax = axs[0]
data_violin = [mono_df[mono_df['metal'] == m]['delta_high_minus_low'].values
               for m in metals_sorted]
vp = ax.violinplot(data_violin, positions=range(len(metals_sorted)),
                   showmedians=True, showextrema=True)
for i, (body, color) in enumerate(zip(vp['bodies'], PALETTE[:len(metals_sorted)])):
    body.set_facecolor(color)
    body.set_alpha(0.6)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(range(len(metals_sorted)))
ax.set_xticklabels([m.upper() for m in metals_sorted], fontsize=8)
ax.set_xlabel('USGS metal', fontsize=9)
ax.set_ylabel('Δρ (high − low contamination)', fontsize=9)
ax.set_title('ρ change: low → high contamination', fontsize=10)
grid_h(ax)

# Panel B: % strictly monotone per metal
ax = axs[1]
pct_mono = [100 * mono_df[mono_df['metal'] == m]['strictly_monotone'].mean()
            for m in metals_sorted]
bars = ax.bar(range(len(metals_sorted)), pct_mono,
              color=PALETTE[:len(metals_sorted)], edgecolor='k', linewidth=0.5)
ax.axhline(33.3, color='gray', lw=0.8, ls='--', label='Chance (33%)')
ax.set_xticks(range(len(metals_sorted)))
ax.set_xticklabels([m.upper() for m in metals_sorted], fontsize=8)
ax.set_xlabel('USGS metal', fontsize=9)
ax.set_ylabel('% control combos with ρ_low < ρ_mid < ρ_high', fontsize=9)
ax.set_title('% strictly monotone across 12 control combos', fontsize=10)
ax.legend(fontsize=8)
grid_h(ax)

fig.suptitle('NB31 Fig 3: Does λ-β ρ increase with contamination level?', y=1.02)
save(fig, FIGS / 'nb31_F3_monotonicity')
print('Saved nb31_F3_monotonicity.pdf')

Saved nb31_F3_monotonicity.pdf


In [12]:
# Figure 4: Scatter λ vs |β| for best metal, showing all 3 strata side-by-side
# Use 'None' controls; pick the metal with the largest Δρ (high − low)

best_metal = (
    mono_df[mono_df['controls'] == 'None']
    .sort_values('delta_high_minus_low', ascending=False)
    .iloc[0]['metal']
)
print(f'Best metal for scatter: {best_metal.upper()} (largest Δρ in None controls)')

CAT_ORDER = ['Resistance/Detoxification', 'Transport/Homeostasis',
             'Cofactor Biosynthesis', 'Sensing/Regulation',
             'Metal-dependent Metabolism', 'Unknown']
CAT_COLORS = dict(zip(CAT_ORDER, PALETTE))

fig, axs = plt.subplots(1, 3, figsize=(FIGW['full'], ROW_H), sharey=True)

for ax, stratum in zip(axs, STRATUM_LABELS):
    sub = sweep_lam[
        (sweep_lam['metal'] == best_metal) &
        (sweep_lam['stratum'] == stratum) &
        (sweep_lam['controls'] == 'None')
    ].copy()

    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['lambda_uncond'], ss['abs_beta'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=30,
                   alpha=0.85, edgecolors='k', linewidths=0.4,
                   label=cat.split('/')[0])

    sig = sub[sub['sig']]
    if len(sig) > 0:
        ax.scatter(sig['lambda_uncond'], sig['abs_beta'],
                   facecolors='none', edgecolors='k', s=80, linewidths=1.2, zorder=5)

    if len(sub) >= 5:
        rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
    else:
        rho, p = np.nan, np.nan

    n_kos = sub['ko_id'].nunique()
    ax.annotate(f'ρ={rho:+.3f}\np={p:.2e}\nn={n_kos}',
                xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel("Unconditional Pagel's λ", fontsize=9)
    ax.set_ylabel('|β_USGS| (no controls)', fontsize=9)
    ax.set_title(f'{stratum.title()} contamination\n({best_metal.upper()})', fontsize=9)
    grid_h(ax)

axs[0].legend(fontsize=7, loc='upper right')

fig.suptitle(f'NB31 Fig 4: {best_metal.upper()} — λ vs |β| by contamination stratum '
             '(rings = p<0.05)', y=1.02)
save(fig, FIGS / 'nb31_F4_best_metal_scatter')
print(f'Saved nb31_F4_best_metal_scatter.pdf')

Best metal for scatter: AS (largest Δρ in None controls)


Saved nb31_F4_best_metal_scatter.pdf


In [13]:
# Text summary of key findings
print('=' * 65)
print('NB31 KEY FINDINGS')
print('=' * 65)

print(f'\n1. Overall monotonicity:')
print(f'   {n_mono}/{n_total} metal×combo combinations show ρ_low < ρ_mid < ρ_high')
print(f'   ({100*n_mono/n_total:.1f}% vs 33.3% by chance)')
print(f'   Mean Δρ (high − low): {mean_d:+.3f}')

print(f'\n2. Δρ per metal (None controls → full panel):')
print(f'   {"Metal":5s}  {"Δρ(None)":>10s}  {"Δρ(full)":>10s}  {"Direction"}')
print('   ' + '-' * 48)
for metal in sorted(USGS_METALS):
    d_none = mono_df[(mono_df['metal'] == metal) &
                     (mono_df['controls'] == 'None')]['delta_high_minus_low'].values
    d_full = mono_df[(mono_df['metal'] == metal) &
                     (mono_df['controls'] == 'pH + SOM + Temp + |Lat|')]['delta_high_minus_low'].values
    if len(d_none) > 0 and len(d_full) > 0:
        direction = 'HIGHER in high-contam' if d_none[0] > 0 else 'LOWER in high-contam'
        print(f'   {metal.upper():5s}  {d_none[0]:>+10.3f}  {d_full[0]:>+10.3f}  {direction}')

print(f'\n3. Most significant ρ values (p<0.05):')
sig_rho = summary[summary['p_value'] < 0.05].sort_values('rho', ascending=False)
for _, r in sig_rho.head(10).iterrows():
    print(f'   {r["metal"].upper():3s} | {r["stratum"]:4s} | {r["controls"]:<26s}: '
          f'ρ={r["rho"]:+.3f} p={r["p_value"]:.3e} n={r["n_kos"]}')

print('\n[End of NB31]')

NB31 KEY FINDINGS

1. Overall monotonicity:
   9/84 metal×combo combinations show ρ_low < ρ_mid < ρ_high
   (10.7% vs 33.3% by chance)
   Mean Δρ (high − low): -0.156

2. Δρ per metal (None controls → full panel):
   Metal    Δρ(None)    Δρ(full)  Direction
   ------------------------------------------------
   AS         +0.579      +0.408  HIGHER in high-contam
   CD         -0.630      -0.424  LOWER in high-contam
   CR         +0.002      -0.059  HIGHER in high-contam
   CU         -0.177      -0.722  LOWER in high-contam
   NI         -0.688      -0.100  LOWER in high-contam
   PB         -0.257      -0.343  LOWER in high-contam
   ZN         +0.425      -0.086  HIGHER in high-contam

3. Most significant ρ values (p<0.05):
   PB  | low  | pH + Temp                 : ρ=+0.723 p=4.720e-04 n=19
   PB  | low  | pH + |Lat|                : ρ=+0.695 p=9.630e-04 n=19
   ZN  | high | SOM                       : ρ=+0.640 p=1.770e-03 n=21
   ZN  | high | pH + SOM + |Lat|          : ρ=+0.618